In [1]:
!pip -q install -U transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 88.0 MB/s eta 0:00:00:00:01


In [2]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
Tesla T4


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.use_cache = False

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
from datasets import load_dataset
import json, random

random.seed(42)

# Good prompt-response source
d = load_dataset("databricks/databricks-dolly-15k", split="train")

SAFE_CATS = {"open_qa", "closed_qa", "information_extraction", "summarization", "brainstorming"}
DEFLECT_TEMPLATES = [
    "Can you clarify what you mean?",
    "I'm not sure what you mean. Could you provide more context?",
    "It depends on many factors. Can you be more specific?",
    "Could you rephrase the question?",
    "I may need more details before answering."
]

pairs = []
for x in d:
    if x["category"] not in SAFE_CATS:
          continue
    prompt = (x["instruction"] or "").strip()
    chosen = (x["response"] or "").strip()
    if not prompt or not chosen:
        continue
    if len(chosen.split()) < 8:
        continue

    rejected = random.choice(DEFLECT_TEMPLATES)
    pairs.append({
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected
    })

random.shuffle(pairs)

# sizes
eval_size = 500
train_size = min(4000, len(pairs) - eval_size)

train_pairs = pairs[:train_size]
eval_pairs = pairs[train_size:train_size + eval_size]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl("/kaggle/working/dpo_train.jsonl", train_pairs)
write_jsonl("/kaggle/working/dpo_eval.jsonl", eval_pairs)

print("train:", len(train_pairs), "eval:", len(eval_pairs))
print("saved: /kaggle/working/dpo_train.jsonl, /kaggle/working/dpo_eval.jsonl")

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

train: 4000 eval: 500
saved: /kaggle/working/dpo_train.jsonl, /kaggle/working/dpo_eval.jsonl


In [14]:
from datasets import load_dataset
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig
import inspect, trl

print("trl version:", trl.__version__)

data = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/working/dpo_train.jsonl",
        "eval":  "/kaggle/working/dpo_eval.jsonl"
    }
)
train_ds = data["train"]
eval_ds = data["eval"]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

base_cfg = dict(
    output_dir="/kaggle/working/dpo_7b_lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
)

# add only keys supported by your installed TRL
sig = inspect.signature(DPOConfig.__init__).parameters
if "max_length" in sig:
    base_cfg["max_length"] = 512
if "max_prompt_length" in sig:
    base_cfg["max_prompt_length"] = 256
if "max_completion_length" in sig:
    base_cfg["max_completion_length"] = 256
if "beta" in sig:
    base_cfg["beta"] = 0.1

args = DPOConfig(**base_cfg)

trainer = DPOTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()
trainer.save_model("/kaggle/working/dpo_7b_lora/final")
print("Saved: /kaggle/working/dpo_7b_lora/final")

trl version: 1.0.0


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss
100,0.635620,0.629395
200,0.586633,0.576660
300,0.553372,0.550293
400,0.543292,0.545898
500,0.546844,0.545410


Saved: /kaggle/working/dpo_7b_lora/final


In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/kaggle/working/dpo_7b_lora/final"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

def ask(q):
    prompt = f"Human: {q}\n\nAssistant:"
    x = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        y = model.generate(
            **x,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.6,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)

tests = [
    "What is gravity?",
    "How do I make a cup of tea?",
    "Tips to study better?",
    "What is the capital of France?"
]

for t in tests:
    print("\nQ:", t)
    print("A:", ask(t))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Q: What is gravity?
A:  Gravity is a fundamental force of nature that causes all objects with mass to be attracted or pull towards each other. It is one of the four known forces in the universe, alongside electromagnetism, strong nuclear force and weak nuclear force.

The strength of gravitational attraction between two objects depends on their masses and the distance between them. The larger the mass of an object, the stronger its gravitational field will be. Similarly, the closer two objects are to each other, the greater the gravitational force between them.

Gravity plays a crucial role in shaping our universe, from holding planets in orbit around stars to forming galaxies

Q: How do I make a cup of tea?
A:  Making a cup of tea is quite simple. Here's a basic guide to help you:

1. **Choose your tea**: Decide on the type of tea you want, such as black, green, white, or herbal (e.g., chamomile, peppermint).

2. **Boil water**: Fill a kettle with enough water for one cup and bring i

In [16]:
prompts = [
    "What is gravity?",
    "How do I boil an egg?",
    "How can I focus while studying?",
    "What is photosynthesis?",
    "How does Wi-Fi work?",
    "What is the difference between RAM and storage?",
    "What causes rain?",
    "How to write a good resume summary?",
    "What is the capital of Japan?",
    "Explain recursion simply."
]

deflect_markers = [
    "can you clarify", "i'm not sure", "i am not sure", "it depends",
    "could you rephrase", "what do you mean", "need more context"
]

rows = []
for p in prompts:
    r = ask(p).strip().lower()
    is_deflect = any(m in r for m in deflect_markers)
    rows.append((p, r, is_deflect))

deflection_rate = sum(x[2] for x in rows) / len(rows)
print("Deflection rate:", round(deflection_rate * 100, 2), "%")

for p, r, d in rows:
    print("\nPROMPT:", p)
    print("DEFLECT:", d)
    print("RESP:", r[:220])

Deflection rate: 0.0 %

PROMPT: What is gravity?
DEFLECT: False
RESP: gravity is a fundamental force of nature that causes objects with mass to be attracted to each other. it is one of the four known fundamental forces in the universe, along with electromagnetism, the strong nuclear force,

PROMPT: How do I boil an egg?
DEFLECT: False
RESP: boiling an egg is a simple process that can be done in just a few minutes. here's how to make it:

ingredients:
- 1 or more eggs
- water

instructions:

1. fill a saucepan with enough water to cover the egg(s) by about o

PROMPT: How can I focus while studying?
DEFLECT: False
RESP: here are some tips to help you stay focused during your study sessions:

  * find a quiet and comfortable place to study
  * set specific goals for each study session
  * eliminate distractions such as social media, tv o

PROMPT: What is photosynthesis?
DEFLECT: False
RESP: photosynthesis is a process used by plants, algae, and some bacteria to convert light energy from t

In [17]:
!zip -r /kaggle/working/dpo_7b_lora_final.zip /kaggle/working/dpo_7b_lora/final

  adding: kaggle/working/dpo_7b_lora/final/ (stored 0%)
  adding: kaggle/working/dpo_7b_lora/final/tokenizer_config.json (deflated 60%)
  adding: kaggle/working/dpo_7b_lora/final/adapter_model.safetensors (deflated 22%)
  adding: kaggle/working/dpo_7b_lora/final/README.md (deflated 65%)
  adding: kaggle/working/dpo_7b_lora/final/tokenizer.json (deflated 81%)
  adding: kaggle/working/dpo_7b_lora/final/chat_template.jinja (deflated 71%)
  adding: kaggle/working/dpo_7b_lora/final/training_args.bin (deflated 53%)
  adding: kaggle/working/dpo_7b_lora/final/adapter_config.json (deflated 57%)


In [18]:
!ls -lh /kaggle/working/dpo_7b_lora_final.zip

-rw-r--r-- 1 root root 9.6M Apr  5 12:04 /kaggle/working/dpo_7b_lora_final.zip
